In [0]:
from dataclasses import dataclass, field
from typing import Optional
import hashlib
import json as _json
import pandas as pd
import numpy as np


# ============================================================================
# 1. MODEL + PURE DERIVATIONS (from engine_config)
# ============================================================================
@dataclass
class TableSpec:
    name: str
    group: str                                  # "core" | "reference" (EXPLICIT; was `generator`)
    pk: Optional[str] = None                    # None => pure-linked table (death-style)
    id_type: str = "long"
    context_fk: Optional[tuple] = None          # (column, parent_table) — the ONE modeled edge
    owner_path: Optional[list] = None           # [(col,parent),...] to privacy_unit; None=derive
    ref_fks: list = field(default_factory=list)
    internal_ref_parent: dict = field(default_factory=dict)  # ref_fk col -> parent table
    self_fk: Optional[str] = None
    drop_cols: list = field(default_factory=list)            # excluded from training entirely
    date_cols: list = field(default_factory=list)
    source_table: Optional[str] = None          # for table-split slices
    where: Optional[str] = None
    split_group: Optional[str] = None           # slices sharing this UNION back to it on write
    parent_window: Optional[dict] = None        # {parent_start, parent_end, child_cols:[...]}
    base_rate_table: bool = False               # 0/1-per-privacy_unit -> C2 calibration eligible
    caps: Optional[dict] = None                 # {per_parent, per_subject}
    encoding_overrides: dict = field(default_factory=dict)

    @property
    def generator(self):                        # legacy alias: profiler hash payload reads spec.generator
        return self.group


@dataclass
class DatasetConfig:
    name: str
    source: dict
    target: dict
    subject: str
    tables: dict
    dp: dict
    profiler: dict
    rare: dict
    privacy_unit: Optional[str] = None          # default = subject
    valprot_override: dict = field(default_factory=dict)
    disable_valprot_tables: list = field(default_factory=list)
    reference_chain: Optional[list] = None
    lifespan_rule: Optional[dict] = None
    pii: dict = field(default_factory=dict)     # table -> [cols]
    freetext_topk: list = field(default_factory=list)
    pk_band_size: int = 100_000_000
    table_order: Optional[list] = None          # explicit topo order; None -> depth-sort
    model_limits: dict = field(default_factory=lambda: {"max_training_time": 120, "max_epochs": 50})
    enable_model_report: bool = False
    extra: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.privacy_unit is None:
            self.privacy_unit = self.subject


def _depth_order(cfg):
    """Stable topo order via depth-to-root over context_fk + internal_ref_parent edges. Raises on
    CYCLE. A dangling parent (declared but absent from cfg.tables) is treated as a root so a
    MALFORMED config can still be ordered — validate_config reports the missing parent instead."""
    specs = cfg.tables

    def depth(name, seen=()):
        if name in seen:
            raise ValueError(f"cycle in context/ref graph at {name}: {seen + (name,)}")
        if name not in specs:
            return 0          # dangling parent -> root; validate_config flags the break, never crashes
        s = specs[name]
        parents = []
        if s.context_fk:
            parents.append(s.context_fk[1])
        parents += list(s.internal_ref_parent.values())
        return 0 if not parents else 1 + max(depth(p, seen + (name,)) for p in parents)

    return sorted(specs, key=lambda n: (depth(n), n))


def topo_order(cfg, group=None):
    """Topological order over the context tree. cfg.table_order (if set) is used verbatim (lets a
    dataset pin the exact hand-order — OMOP config #1 does, for byte-identical legacy behaviour);
    else depth-sort. Optionally filter to a group (core|reference)."""
    if cfg.table_order is not None:
        order = list(cfg.table_order)
        _depth_order(cfg)                       # cheap cycle-check even with explicit order
    else:
        order = _depth_order(cfg)
    if group:
        order = [n for n in order if cfg.tables[n].group == group]
    return order


_ec_topo = topo_order                           # internal alias (kept from the shim era; harmless)


def pk_id_base(cfg):
    return {name: (i + 1) * cfg.pk_band_size for i, name in enumerate(_ec_topo(cfg))}


SDTYPE_TO_ENCODING = {
    "categorical": "TABULAR_CATEGORICAL",
    "numerical": "TABULAR_NUMERIC_AUTO",
    "integer": "TABULAR_NUMERIC_DISCRETE",
    "datetime": "TABULAR_DATETIME",
    "boolean": "TABULAR_CATEGORICAL",
    "id": "TABULAR_CATEGORICAL",
}


def encoding_for(cfg, table, column, sdtype):
    """*_concept_id -> categorical; *_source_value/declared freetext -> categorical; sdtype map; else AUTO."""
    if column.endswith("_concept_id"):
        return "TABULAR_CATEGORICAL"
    if column in cfg.freetext_topk or column.endswith("_source_value"):
        return "TABULAR_CATEGORICAL"
    if sdtype and sdtype in SDTYPE_TO_ENCODING:
        return SDTYPE_TO_ENCODING[sdtype]
    return "AUTO"


_ec_enc = encoding_for


def trained_columns(cfg, table, all_columns):
    """Columns the model TRAINS on: drop pk, context fk, self fk, ref fks, pii, drop_cols, and —
    for a grandchild table (context parent is NOT the subject) — the subject key, re-derived in
    assembly."""
    s = cfg.tables[table]
    drop = set(s.drop_cols) | set(s.ref_fks) | set(cfg.pii.get(table, []))
    if s.pk:
        drop.add(s.pk)
    if s.context_fk:
        drop.add(s.context_fk[0])
    if s.self_fk:
        drop.add(s.self_fk)
    if s.context_fk and s.context_fk[1] != cfg.subject:
        drop.add(f"{cfg.subject}_id")
    return [c for c in all_columns if c not in drop]


_ec_tc = trained_columns


def frame_columns(cfg, table, all_columns):
    """trained_columns + pk + context fk (MOSTLY AI needs the key cols present in `columns`)."""
    s = cfg.tables[table]
    cols = set(_ec_tc(cfg, table, all_columns))
    if s.pk:
        cols.add(s.pk)
    if s.context_fk:
        cols.add(s.context_fk[0])
    return [c for c in all_columns if c in cols]


def dp_block(cfg):
    dp = cfg.dp
    if not dp.get("enabled"):
        return None
    return {"max_epsilon": dp["max_epsilon"],
            "value_protection_epsilon": dp["value_protection_epsilon"],
            "delta": dp["delta"], "noise_multiplier": dp["noise_multiplier"],
            "max_grad_norm": dp["max_grad_norm"]}


_ec_dp_block = dp_block


def build_table_config(cfg, table, df, sdtypes_for_table):
    """One MOSTLY AI SourceTableConfig dict. DP only for core tables; value_protection=False for
    cfg.disable_valprot_tables; per-table limits capped."""
    s = cfg.tables[table]
    cols_in_df = list(df.columns)
    key_cols = set()
    if s.pk:
        key_cols.add(s.pk)
    if s.context_fk:
        key_cols.add(s.context_fk[0])
    columns = []
    for c in cols_in_df:
        if c in key_cols:
            columns.append({"name": c, "model_encoding_type": "AUTO"})
            continue
        enc = _ec_enc(cfg, table, c, sdtypes_for_table.get(c))
        enc = s.encoding_overrides.get(c, enc)
        columns.append({"name": c, "model_encoding_type": enc})

    out = {"name": s.name, "data": df}
    if s.pk:
        out["primary_key"] = s.pk
    tmc = {"enable_model_report": cfg.enable_model_report,
           "max_training_time": cfg.model_limits["max_training_time"],
           "max_epochs": cfg.model_limits["max_epochs"]}
    if s.group == "core" and s.name in cfg.disable_valprot_tables:
        tmc["value_protection"] = False
    dp = _ec_dp_block(cfg)
    if dp is not None and s.group == "core":
        tmc["differential_privacy"] = dp
    out["tabular_model_configuration"] = tmc
    if s.context_fk:
        out["foreign_keys"] = [{"column": s.context_fk[0],
                                "referenced_table": s.context_fk[1], "is_context": True}]
    if columns:
        out["columns"] = columns
    return out


def sdtypes_table_key(cfg, table):
    """Key into the sdtypes typing seed for a table. The dataset prefix comes from
    cfg.extra['sdtypes_prefix'] (default 'omop_' for back-compat with config #1)."""
    s = cfg.tables[table]
    base = s.source_table or s.name
    prefix = cfg.extra.get("sdtypes_prefix", "omop_")
    return f"{prefix}{base}"


def validate_config(cfg):
    """Return a list of human-readable config errors ([] == valid). Pure (no Spark). MUST NOT raise
    on any malformed config — every structural defect is appended to `errs` and returned."""
    errs = []
    names = set(cfg.tables)
    if cfg.subject not in names:
        errs.append(f"subject '{cfg.subject}' not in tables {sorted(names)}")
    if cfg.privacy_unit not in names:
        errs.append(f"privacy_unit '{cfg.privacy_unit}' not in tables")
    for n, s in cfg.tables.items():
        if s.group not in ("core", "reference"):
            errs.append(f"{n}: group '{s.group}' not in (core, reference)")
        if s.context_fk and s.context_fk[1] not in names:
            errs.append(f"{n}: context_fk parent '{s.context_fk[1]}' MISSING from tables")
        if s.pk and s.context_fk and s.pk == s.context_fk[0]:
            errs.append(f"{n}: pk and context_fk share column {s.pk}")
        for c, p in s.internal_ref_parent.items():
            if p not in names:
                errs.append(f"{n}: internal_ref_parent parent '{p}' MISSING")
        if s.parent_window:
            pw = s.parent_window
            if not all(k in pw for k in ("parent_start", "parent_end", "child_cols")):
                errs.append(f"{n}: parent_window must have parent_start, parent_end, child_cols")
    for n in (cfg.reference_chain or []):
        if n not in names:
            errs.append(f"reference_chain table '{n}' MISSING")
    for n in cfg.disable_valprot_tables:
        if n not in names:
            errs.append(f"disable_valprot_tables '{n}' MISSING")
    for n in cfg.valprot_override:
        if n not in names:
            errs.append(f"valprot_override '{n}' MISSING")
    try:
        _ec_topo(cfg)                           # cycle probe; tolerant of dangling parents (root)
    except (ValueError, KeyError) as e:         # NEVER let the probe escape — report and continue
        errs.append(f"topo_order error: {e}")
    return errs


def dataset_config_hash(cfg):
    """Stable 16-hex hash of the dataset's structural config (DP + profiler + graph shape + order +
    subject/privacy_unit). RENAMED from the P1-era engine `config_hash(cfg)` so it cannot collide
    with the 0-arg PROFILER-CONTRACT `config_hash()` below (different payload, different 16-hex)."""
    shape = {}
    for name in sorted(cfg.tables):
        s = cfg.tables[name]
        shape[name] = {"group": s.group, "pk": s.pk, "context_fk": s.context_fk,
                       "source_table": s.source_table, "where": s.where, "self_fk": s.self_fk,
                       "ref_fks": sorted(s.ref_fks), "drop_cols": sorted(s.drop_cols),
                       "date_cols": sorted(s.date_cols)}
    payload = {"dp": {k: cfg.dp[k] for k in sorted(cfg.dp)},
               "profiler": {k: cfg.profiler[k] for k in sorted(cfg.profiler)},
               "graph_shape": shape, "core_order": _ec_topo(cfg, "core"),
               "subject": cfg.subject, "privacy_unit": cfg.privacy_unit}
    return hashlib.sha256(_json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


# ============================================================================
# 2. INTROSPECT — schema -> column roles (from the introspect module)
# ============================================================================
def roles_from_schema(cfg, table, colnames, sdtypes_for_table):
    """PURE: {col: encoding_role} for a table, given its column NAMES + an sdtype map.
    Applies per-table encoding_overrides first, then encoding_for. AUTO == ambiguous/safe default."""
    s = cfg.tables[table]
    roles = {}
    for c in colnames:
        roles[c] = s.encoding_overrides.get(c) or _ec_enc(cfg, table, c, sdtypes_for_table.get(c))
    return roles


def ambiguous_cols(roles):
    """Columns that resolved to AUTO (no confident role) — surfaced for review."""
    return [c for c, r in roles.items() if r == "AUTO"]


def introspect_roles(cfg, spark, load_sdtypes=None):
    """SPARK: read each declared table's schema from cfg.source and return {table: {col: role}}.
    `load_sdtypes` (optional) returns {table_key: {col: sdtype}}; if None, roles come from name
    heuristics only. Logs ambiguous cols."""
    src = f"{cfg.source['catalog']}.{cfg.source['schema']}"
    prefix = cfg.extra.get("sdtypes_prefix", "omop_")
    sdt = load_sdtypes() if load_sdtypes else {}
    out = {}
    for t, s in cfg.tables.items():
        base = s.source_table or t
        try:
            cols = [f.name for f in spark.table(f"{src}.{base}").schema.fields]
        except Exception as e:
            print(f"INTROSPECT {t}: cannot read {src}.{base} ({e}) — skipped")
            out[t] = {}
            continue
        roles = roles_from_schema(cfg, t, cols, sdt.get(f"{prefix}{base}", {}))
        amb = ambiguous_cols(roles)
        if amb:
            print(f"INTROSPECT {t}: {len(amb)}/{len(roles)} AUTO/ambiguous -> safe default: {amb[:10]}")
        out[t] = roles
    return out


# ============================================================================
# 3. REKEY — pure-function PK re-minting
# ============================================================================
def _rng(seed):
    return np.random.default_rng(int(seed))


def rekey(df: pd.DataFrame, pk: str, base: int) -> tuple:
    """Mint dense BIGINT keys for `df[pk]`, starting at `base`.
    Returns (keyed_df, mapping); mapping is DataFrame[old_<pk>, <pk>]."""
    old_col = f"old_{pk}"
    n = len(df)
    out = df.copy()
    if n == 0:
        mapping = pd.DataFrame({old_col: pd.Series([], dtype="int64"),
                                pk: pd.Series([], dtype="int64")})
        return out, mapping
    new_ids = pd.Series(range(base, base + n), index=out.index, dtype="int64")
    mapping = pd.DataFrame({old_col: out[pk].to_numpy(), pk: new_ids.to_numpy()})
    out[pk] = new_ids.to_numpy()
    return out, mapping


def remap_fk(child: pd.DataFrame, fk: str, mapping: pd.DataFrame, parent_pk: str,
             how: str = "inner") -> pd.DataFrame:
    """Repoint `child[fk]` old->new via the rekey mapping. inner drops orphans; left -> NaN."""
    old_col = f"old_{parent_pk}"
    m = mapping.rename(columns={old_col: fk, parent_pk: f"__new_{fk}"})
    merged = child.merge(m[[fk, f"__new_{fk}"]], on=fk, how=how)
    merged[fk] = merged[f"__new_{fk}"]
    merged = merged.drop(columns=[f"__new_{fk}"])
    return merged


def derive_parent_key(child: pd.DataFrame, parent_keyed: pd.DataFrame,
                      link_col: str, derived_col: str) -> pd.DataFrame:
    """Derive `derived_col` on child from its already-rekeyed parent (e.g. event.person_id
    FROM the parent visit row, guaranteeing event.person_id == visit.person_id)."""
    cols = [link_col, derived_col]
    out = child.merge(parent_keyed[cols], on=link_col, how="left",
                      suffixes=("", "__from_parent"))
    src = derived_col + "__from_parent"
    if src in out.columns:
        out[derived_col] = out[src]
        out = out.drop(columns=[src])
    return out


# ============================================================================
# 4. TEMPORAL — pure-function date repair
# ============================================================================
PANDAS_MIN = pd.Timestamp.min.ceil("D")   # 1677-09-22
PANDAS_MAX = pd.Timestamp.max.floor("D")  # 2262-04-11
BUSINESS_MIN = pd.Timestamp("1900-01-01")
BUSINESS_MAX = pd.Timestamp("2099-12-31")


def _as_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")


def order_pair(df: pd.DataFrame, start_col: str, end_col: str) -> pd.DataFrame:
    """Ensure end >= start. Where end < start (both present), set end = start. Idempotent."""
    out = df.copy()
    if start_col not in out or end_col not in out:
        return out
    s = _as_dt(out[start_col])
    e = _as_dt(out[end_col])
    bad = s.notna() & e.notna() & (e < s)
    e = e.mask(bad, s)
    out[start_col] = s
    out[end_col] = e
    return out


def clamp_into_window(df: pd.DataFrame, cols, lo: pd.Series, hi: pd.Series) -> pd.DataFrame:
    """Clamp each col into the per-row [lo, hi] window. NaT bounds = no constraint. Idempotent."""
    out = df.copy()
    lo = _as_dt(lo).reindex(out.index)
    hi = _as_dt(hi).reindex(out.index)
    for c in cols:
        if c not in out:
            continue
        v = _as_dt(out[c])
        below = v.notna() & lo.notna() & (v < lo)
        above = v.notna() & hi.notna() & (v > hi)
        v = v.mask(below, lo).mask(above, hi)
        out[c] = v
    return out


def birth_before_events(df: pd.DataFrame, birth_col: str, event_cols) -> pd.DataFrame:
    """Ensure birth <= each event date; events preceding birth are pushed up to birth."""
    out = df.copy()
    if birth_col not in out:
        return out
    b = _as_dt(out[birth_col])
    for c in event_cols:
        if c not in out:
            continue
        v = _as_dt(out[c])
        bad = v.notna() & b.notna() & (v < b)
        out[c] = v.mask(bad, b)
    out[birth_col] = b
    return out


def death_after_events(df: pd.DataFrame, death_col: str, last_event_col: str) -> pd.Series:
    """Boolean mask: death valid (>= last event or unknown). NaT death = valid."""
    d = _as_dt(df[death_col]) if death_col in df else pd.Series(pd.NaT, index=df.index)
    le = _as_dt(df[last_event_col]) if last_event_col in df else pd.Series(pd.NaT, index=df.index)
    return ~(d.notna() & le.notna() & (d < le))


def clamp_valid_range(df: pd.DataFrame, cols, lo: pd.Timestamp = None,
                      hi: pd.Timestamp = None) -> pd.DataFrame:
    """Clamp datetime cols into [lo, hi] (defaults: pandas-representable bounds). Idempotent."""
    lo = PANDAS_MIN if lo is None else lo
    hi = PANDAS_MAX if hi is None else hi
    out = df.copy()
    for c in cols:
        if c not in out:
            continue
        v = _as_dt(out[c])
        v = v.mask(v.notna() & (v < lo), lo)
        v = v.mask(v.notna() & (v > hi), hi)
        out[c] = v
    return out


def rebuild_preceding_chain(visits: pd.DataFrame, person_col: str, pk: str,
                            start_col: str, preceding_col: str) -> pd.DataFrame:
    """Rebuild a valid self-referential preceding chain (per person, sorted by start)."""
    out = visits.copy()
    if out.empty:
        out[preceding_col] = pd.Series(dtype="float64")
        return out
    out["__order"] = _as_dt(out[start_col])
    out = out.sort_values([person_col, "__order"], kind="mergesort")
    prev_pk = out.groupby(person_col, sort=False)[pk].shift(1)
    out[preceding_col] = prev_pk
    out = out.drop(columns="__order")
    return out


def date_from_datetime(df: pd.DataFrame, pairs) -> pd.DataFrame:
    """Re-derive each DATE column from its DATETIME counterpart after clamping."""
    out = df.copy()
    for date_col, dt_col in pairs:
        if date_col in out and dt_col in out:
            dt = _as_dt(out[dt_col])
            out[date_col] = dt.dt.normalize().where(dt.notna(), _as_dt(out[date_col]))
    return out


# ============================================================================
# 5. FK_REMAP — reference-chain tuple assignment (generic over cfg.reference_chain)
# ============================================================================
def chain_keys(cfg) -> list:
    """The tuple key columns: pk of each reference_chain table, leaf->root order."""
    return [cfg.tables[t].pk for t in (cfg.reference_chain or [])]


def build_chain_tuples(cfg, frames: dict) -> pd.DataFrame:
    """Pre-join the (rekeyed) reference-chain tables into a tuple table.
    frames: {table_name: rekeyed DataFrame}. Walks leaf->root via each spec's
    internal_ref_parent (left joins, so dangling parents keep NaN downstream cols)."""
    chain = list(cfg.reference_chain or [])
    if not chain:
        return pd.DataFrame()
    leaf = chain[0]
    leaf_spec = cfg.tables[leaf]
    fk_cols = list(leaf_spec.internal_ref_parent.keys())
    tup = frames[leaf][[leaf_spec.pk] + fk_cols].drop_duplicates(leaf_spec.pk)
    for t in chain[1:]:
        spec = cfg.tables[t]
        up_fks = list(spec.internal_ref_parent.keys())
        seg = frames[t][[spec.pk] + up_fks].drop_duplicates(spec.pk)
        tup = tup.merge(seg, on=spec.pk, how="left")
    return tup[chain_keys(cfg)]


def assign_subject_reference(cfg, subject_df: pd.DataFrame, tuples: pd.DataFrame,
                             seed: int) -> pd.DataFrame:
    """Assign the subject's reference FKs as a CONSISTENT tuple (random tuple row per subject).
    Sets every chain key present in the subject spec's ref_fks."""
    out = subject_df.copy()
    keys = [k for k in chain_keys(cfg) if k in cfg.tables[cfg.subject].ref_fks]
    n = len(out)
    if n == 0 or tuples.empty:
        for c in keys:
            out[c] = pd.Series([pd.NA] * n, index=out.index)
        return out
    rng = _rng(seed)
    idx = rng.integers(0, len(tuples), size=n)
    picked = tuples.iloc[idx].reset_index(drop=True)
    out = out.reset_index(drop=True)
    for c in keys:
        out[c] = picked[c].to_numpy()
    return out


def assign_child_reference(cfg, child: pd.DataFrame, table: str, subject_keyed: pd.DataFrame,
                           tuples: pd.DataFrame, seed: int) -> pd.DataFrame:
    """Assign a hub table's reference FKs consistent WITH ITS SUBJECT: prefer a tuple sharing
    the subject's chain-ROOT key (OMOP: same location); fall back to any tuple. Sets the chain
    keys present in the table spec's ref_fks. child must already carry the rekeyed subject key."""
    spec = cfg.tables[table]
    subj_pk = cfg.tables[cfg.subject].pk
    root_key = chain_keys(cfg)[-1]
    keys = [k for k in chain_keys(cfg) if k in spec.ref_fks]
    out = child.copy()
    if out.empty or tuples.empty:
        for c in keys:
            out[c] = pd.Series([pd.NA] * len(out), index=out.index)
        return out
    rng = _rng(seed)
    sloc = subject_keyed.set_index(subj_pk)[root_key].to_dict()
    by_root = {loc: grp.reset_index(drop=True)
               for loc, grp in tuples.groupby(root_key, sort=False)}
    all_tuples = tuples.reset_index(drop=True)

    picked = {c: [] for c in keys}
    for pid in out[subj_pk].to_numpy():
        loc = sloc.get(pid)
        pool = by_root.get(loc)
        if pool is None or len(pool) == 0:
            pool = all_tuples
        r = pool.iloc[int(rng.integers(0, len(pool)))]
        for c in keys:
            picked[c].append(r[c])
    for c in keys:
        out[c] = picked[c]
    return out


def assign_visit_within_person(event: pd.DataFrame, visit_keyed: pd.DataFrame,
                               seed: int) -> pd.DataFrame:
    """Assign event.visit_occurrence_id from the SAME person's visits (kept, tested, unused —
    the re-root experiment artefact; see memory)."""
    out = event.copy()
    if out.empty:
        out["visit_occurrence_id"] = pd.Series([], dtype="float64")
        return out
    rng = _rng(seed)
    by_person = {pid: grp["visit_occurrence_id"].to_numpy()
                 for pid, grp in visit_keyed.groupby("person_id", sort=False)}
    vals = []
    for pid in out["person_id"].to_numpy():
        pool = by_person.get(pid)
        if pool is None or len(pool) == 0:
            vals.append(np.nan)
        else:
            vals.append(pool[int(rng.integers(0, len(pool)))])
    out["visit_occurrence_id"] = vals
    return out


def inherit_parent_col(event: pd.DataFrame, parent_keyed: pd.DataFrame,
                       link_col: str, col: str) -> pd.DataFrame:
    """Event `col` inherits from its parent row via `link_col` (left join; NaN link -> NaN col)."""
    out = event.copy()
    vp = parent_keyed[[link_col, col]].rename(columns={col: "__parent_col"})
    out = out.merge(vp, on=link_col, how="left")
    out[col] = out["__parent_col"]
    out = out.drop(columns="__parent_col")
    return out


def reinject_null_fk(df: pd.DataFrame, col: str, null_rate: float, seed: int) -> pd.DataFrame:
    """Set a `null_rate` fraction of `col` to NaN (matches a measured source NULL rate)."""
    out = df.copy()
    n = len(out)
    if n == 0 or null_rate <= 0 or col not in out:
        return out
    k = int(round(n * null_rate))
    if k <= 0:
        return out
    rng = _rng(seed)
    pick = rng.choice(n, size=min(k, n), replace=False)
    vals = out[col].to_numpy(dtype="object")
    vals[pick] = np.nan
    out[col] = vals
    return out


def cap_per_visit(df: pd.DataFrame, fk: str, cap: int, seed: int) -> pd.DataFrame:
    """Keep at most `cap` rows per fk value (seeded deterministic shuffle). NULL-fk rows kept."""
    out = df.copy()
    n = len(out)
    if n == 0 or cap is None or cap <= 0 or fk not in out.columns:
        return out
    rng = _rng(seed)
    order = rng.permutation(n)
    out = out.iloc[order].reset_index(drop=True)
    notnull = out[fk].notna()
    rank = out[notnull].groupby(fk, sort=False).cumcount()
    keep_idx = rank[rank < cap].index
    keep_mask = ~notnull
    keep_mask.loc[keep_idx] = True
    capped = out[keep_mask].reset_index(drop=True)
    return capped


def calibrate_rate(df: pd.DataFrame, n_parents: int, target_rate: float, seed: int,
                   tol: float, allow_upsample: bool = False) -> pd.DataFrame:
    """Calibrate a 0/1-per-parent table to a target rate by DOWN-SAMPLING (never inventing
    rows). No-op if empty / n_parents<=0 / within tol / under target (unless allow_upsample)."""
    n = len(df)
    if n == 0 or n_parents <= 0:
        return df.copy()
    current = n / n_parents
    if abs(current - target_rate) <= tol:
        return df.copy()
    keep = int(round(target_rate * n_parents))
    if current > target_rate:
        keep = max(0, min(keep, n))
        rng = _rng(seed)
        idx = sorted(rng.permutation(n)[:keep])
        return df.iloc[idx].reset_index(drop=True)
    if not allow_upsample:
        return df.copy()
    rng = _rng(seed)
    extra = max(0, keep - n)
    dup = df.iloc[rng.integers(0, n, size=extra)] if extra else df.iloc[0:0]
    return pd.concat([df, dup], ignore_index=True)


def orphan_count(child: pd.DataFrame, fk: str, parent: pd.DataFrame, pk: str) -> int:
    """Child rows whose non-null fk has no matching parent pk."""
    if fk not in child or child.empty:
        return 0
    fkvals = child[fk].dropna()
    if fkvals.empty:
        return 0
    valid = set(parent[pk].dropna().tolist())
    return int((~fkvals.isin(valid)).sum())


def tuple_consistency_violations(assigned: pd.DataFrame, tuples: pd.DataFrame,
                                 keys=("provider_id", "care_site_id", "location_id")) -> int:
    """Rows in `assigned` whose chain-key combination does NOT exist in `tuples`.
    Rows with any NaN key are skipped (tolerated). Pass keys=chain_keys(cfg) for cfg-first use."""
    keys = list(keys)
    sub = assigned[keys].dropna()
    if sub.empty:
        return 0
    valid = set(map(tuple, tuples[keys].dropna().itertuples(index=False, name=None)))
    got = map(tuple, sub.itertuples(index=False, name=None))
    return sum(1 for t in got if t not in valid)


# ============================================================================
# 6. RARITY — DP-accounted profiler + C1 collapse policy, cfg-first.
#    HASH CONTRACT: config_hash() payload identical to the legacy rarity module
#    ('generator' via spec.generator, 'schema': RARITY_SCHEMA_VERSION) -> same 16-hex for the
#    same dataset (OMOP: d3c547df7d928464). All functions read `cfg` from globals AT CALL TIME.
#    NOTE: consumers that want the PROFILER/RARE/VALPROT_OVERRIDE dicts bind them in their own
#    cfg cell (PROFILER = cfg.profiler; RARE = cfg.rare; VALPROT_OVERRIDE = cfg.valprot_override)
#    — this notebook does NOT read cfg at %run time, so %run order vs the dataset config is free.
# ============================================================================
RARITY_SCHEMA_VERSION = 1


def det_seed(base, key):
    """Deterministic per-(base,key) seed via full-width sha256 (builtin hash() is salted)."""
    h = hashlib.sha256(f"{int(base)}|{key}".encode()).hexdigest()
    return int(h, 16)


def legacy_valprot_off():
    """The known-safe force-off set (the proven sparse tables) from cfg."""
    return list(cfg.disable_valprot_tables)


def _graph_shape():
    """Structural shape of every table spec the profiler depends on (key 'generator' kept for
    hash compatibility; value == spec.group via the legacy alias property)."""
    shape = {}
    for name in sorted(cfg.tables):
        s = cfg.tables[name]
        shape[name] = {
            "generator": s.generator,
            "pk": s.pk,
            "context_fk": s.context_fk,
            "source_table": s.source_table,
            "where": s.where,
            "self_fk": s.self_fk,
            "ref_fks": sorted(s.ref_fks),
            "drop_cols": sorted(s.drop_cols),
            "date_cols": sorted(s.date_cols),
        }
    return shape


def config_hash():
    """Stable 16-hex hash of the PROFILE-GENERATING config (DP + PROFILER + GRAPH shape + core
    order + schema). EXCLUDES RARE / VALPROT_OVERRIDE (resolution-time knobs). This is the
    freshness token stamped into rarity_profile.json."""
    payload = {
        "dp": {k: cfg.dp[k] for k in sorted(cfg.dp)},
        "profiler": {k: cfg.profiler[k] for k in sorted(cfg.profiler)},
        "graph_shape": _graph_shape(),
        "core_order": _ec_topo(cfg, "core"),
        "schema": RARITY_SCHEMA_VERSION,
    }
    return hashlib.sha256(_json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


RARITY_CONFIG_HASH = config_hash        # shadow-proof handle (P2 contract, kept)


def composed_epsilon(value_protection_on_tables):
    """Honest composed naive epsilon incl. the profiler budget."""
    n_core = len(_ec_topo(cfg, "core"))
    train = cfg.dp["max_epsilon"] * n_core if cfg.dp["enabled"] else 0.0
    valprot = (cfg.dp["value_protection_epsilon"] * len(list(value_protection_on_tables))
               if cfg.dp["enabled"] else 0.0)
    prof = cfg.profiler["epsilon_profiler"] if cfg.dp["enabled"] else 0.0
    return {"train": train, "valprot": valprot, "profiler": prof, "total": train + valprot + prof}


def seqlen_stat(child, parents, parent_key, patient_key, bins, parent_cap):
    """RAW per-parent sequence-length stats (per-patient parent cap bounds sensitivity)."""
    per_parent = child.groupby(parent_key).size()
    p = parents[[parent_key]].drop_duplicates(parent_key).copy()
    if patient_key == parent_key:
        p[patient_key + "__pat"] = p[parent_key]
        pat = patient_key + "__pat"
    else:
        owner = parents[[parent_key, patient_key]].drop_duplicates(parent_key).set_index(parent_key)[patient_key]
        p = p.set_index(parent_key)
        p[patient_key] = owner
        p = p.reset_index()
        pat = patient_key
    p = p.set_index(parent_key)
    p["n"] = per_parent.reindex(p.index).fillna(0).astype(int)
    p = p.sort_index().groupby(pat, sort=False).head(parent_cap)
    vals = p["n"].to_numpy()
    clamped = np.minimum(vals, bins[-1]) if len(vals) else vals
    bin_counts = list(np.histogram(clamped, bins=bins)[0].astype(int))
    frac = float((vals > 0).mean()) if len(vals) else 0.0
    return {"n_parents": int(len(vals)), "frac_parent_with_child": frac,
            "bin_counts": [int(x) for x in bin_counts], "bins": list(bins)}


def base_rate_stat(child, parents, patient_key):
    """RAW positive-rate of a 0/1-per-parent table (sensitivity 1)."""
    n = int(parents[patient_key].nunique())
    pos = int(child[patient_key].nunique())
    return {"n_parents": n, "pos_count": pos, "rate": (pos / n) if n else 0.0}


def category_support_stat(rows, cat_col, patient_key, per_patient_cap):
    """RAW distinct-PATIENT support per category (per-patient cap bounds sensitivity)."""
    df = rows[[patient_key, cat_col]].dropna().drop_duplicates()
    df = df.sort_values(cat_col).groupby(patient_key, sort=False).head(per_patient_cap)
    support = df.groupby(cat_col)[patient_key].nunique().to_dict()
    return {"support": {str(k): int(v) for k, v in support.items()},
            "cardinality": int(rows[cat_col].dropna().nunique())}


def noise_count(count, sensitivity, eps, seed):
    """Laplace(sensitivity/eps) on a count; clamped >=0; rounded."""
    if eps is None or eps <= 0:
        return int(max(0, round(count)))
    lap = _rng(seed).laplace(0.0, float(sensitivity) / float(eps))
    return int(max(0, round(count + lap)))


def noise_rate(rate, n, sensitivity, eps, seed):
    """Noise a rate via its positive count; clamp [0,1]."""
    if n <= 0:
        return 0.0
    nc = noise_count(rate * n, sensitivity, eps, seed)
    return float(min(1.0, max(0.0, nc / n)))


def noise_bins(bin_counts, sensitivity, eps, seed):
    """Independent Laplace per histogram bin (distinct det_seed per bin); clamp >=0."""
    return [noise_count(c, sensitivity, eps, det_seed(seed, f"bin{i}")) for i, c in enumerate(bin_counts)]


def noise_support(support, sensitivity, eps, seed_base):
    """Noise each category's support with a deterministic per-category seed."""
    return {k: noise_count(v, sensitivity, eps, det_seed(seed_base, k)) for k, v in support.items()}


def profile_header(mode, cohort_counts, cfg_hash, seed, eps_profiler, source_version):
    return {"schema_version": RARITY_SCHEMA_VERSION, "cohort_mode": mode,
            "cohort_counts": cohort_counts, "config_hash": cfg_hash, "seed": seed,
            "epsilon_profiler": eps_profiler, "source_version": source_version,
            "noise_mechanism": "laplace_bounded_sensitivity"}


def profile_is_fresh(profile, expected_cfg_hash, expected_mode):
    h = (profile or {}).get("header", {})
    return (h.get("schema_version") == RARITY_SCHEMA_VERSION
            and h.get("config_hash") == expected_cfg_hash
            and h.get("cohort_mode") == expected_mode)


def _collapse_prone(seqlen, support_min):
    """True if NO nonzero-length bin has distinct-parent support >= support_min."""
    n = seqlen.get("n_parents", 0)
    if not n:
        return True
    bins = seqlen.get("bins", [])
    counts = seqlen.get("bin_counts", [])
    best = 0.0
    for i, c in enumerate(counts):
        left = bins[i] if i < len(bins) else 0
        if left >= 1:
            best = max(best, c / n)
    return best < support_min


def collapse_policy(table, profile_tables, override, legacy_off, support_min):
    """Resolve value_protection for one table: override > profile > legacy default."""
    if table in override:
        return {"value_protection": bool(override[table]), "source": "override"}
    ent = (profile_tables or {}).get(table)
    if ent and "seqlen" in ent:
        prone = _collapse_prone(ent["seqlen"], support_min)
        return {"value_protection": (not prone), "source": "profile"}
    return {"value_protection": (table not in legacy_off), "source": "legacy_default"}


def resolve_all_policies(core_tables, profile_tables, override, legacy_off, support_min):
    """Resolve collapse policy for every core table."""
    return {t: collapse_policy(t, profile_tables, override, legacy_off, support_min)
            for t in core_tables}


# ============================================================================
# 7. SYNTH_IO — cohort -> pandas + sdtypes seed + MOSTLY AI config build, cfg-first.
# ============================================================================
def load_sdtypes() -> dict:
    """Seed encodings/PII from the dataset's sdtypes table (cfg.extra['sdtypes_table'],
    keys filtered by cfg.extra['sdtypes_prefix']). Returns {table_key: {col: sd_type}}."""
    src = cfg.extra.get("sdtypes_table", "6_mgmt.default.sdtypes")
    prefix = cfg.extra.get("sdtypes_prefix", "omop_")
    try:
        rows = spark.sql(
            f"SELECT table_name, column_name, sd_type FROM {src} "
            f"WHERE table_name LIKE '{prefix}%'").collect()
    except Exception as e:
        print(f"WARN: could not read sdtypes ({e}); encodings fall back to AUTO/heuristics")
        return {}
    out = {}
    for r in rows:
        out.setdefault(r["table_name"], {})[r["column_name"]] = r["sd_type"]
    return out


def cohort_to_pandas(table: str, frame_cols: list) -> pd.DataFrame:
    """Load <target>.cohort_<table>, select frame_cols, return pandas (dates pre-clamped)."""
    tgt = f"{cfg.target['catalog']}.{cfg.target['schema']}"
    sdf = spark.table(f"{tgt}.cohort_{table}").select(*[f"`{c}`" for c in frame_cols])
    return sdf.toPandas()


def enforce_input_ref_integrity(child_pdf: pd.DataFrame, fk: str,
                                parent_pdf: pd.DataFrame, parent_pk: str) -> pd.DataFrame:
    """Drop child rows whose non-null fk has no matching parent pk (clean relational frame)."""
    if fk not in child_pdf.columns or child_pdf.empty:
        return child_pdf
    valid = set(parent_pdf[parent_pk].dropna().tolist())
    keep = child_pdf[fk].isna() | child_pdf[fk].isin(valid)
    dropped = int((~keep).sum())
    if dropped:
        print(f"  drop_unknown_references: dropped {dropped} rows on {fk} not in parent {parent_pk}")
    return child_pdf[keep].reset_index(drop=True)


def drop_null_context_fk(child_pdf: pd.DataFrame, fk_col: str, table_name: str) -> pd.DataFrame:
    """Drop rows with NULL context FK (NULL -> NaN sequence -> pad_ctx_sequences len(float) crash)."""
    null_mask = child_pdf[fk_col].isna()
    n_null = int(null_mask.sum())
    if n_null:
        print(f"  drop_null_context_fk: dropped {n_null} {table_name} rows with null {fk_col} "
              f"({n_null / len(child_pdf):.4%} of {len(child_pdf)})")
    return child_pdf[~null_mask].reset_index(drop=True)


def build_generator_tables(generator: str, sdtypes: dict, frames: dict) -> list:
    """Build the MOSTLY AI `tables` list for a generator group (cfg-first engine calls)."""
    tables = []
    for t in _ec_topo(cfg, generator):
        df = frames[t]
        sd_key = sdtypes_table_key(cfg, t)
        tables.append(build_table_config(cfg, t, df, sdtypes.get(sd_key, {})))
    return tables


def prepare_frames(generator: str) -> dict:
    """Load every cohort table for a generator group into pandas, sliced to frame columns,
    with context-FK referential integrity enforced (orphans + NULLs dropped)."""
    tgt = f"{cfg.target['catalog']}.{cfg.target['schema']}"
    frames = {}
    for t in _ec_topo(cfg, generator):
        all_cols = [f.name for f in spark.table(f"{tgt}.cohort_{t}").schema.fields]
        fcols = frame_columns(cfg, t, all_cols)
        frames[t] = cohort_to_pandas(t, fcols)
        print(f"  loaded {t}: {frames[t].shape}")
    for t in _ec_topo(cfg, generator):
        spec = cfg.tables[t]
        if spec.context_fk:
            fk_col, parent = spec.context_fk
            parent_pk = cfg.tables[parent].pk
            frames[t] = enforce_input_ref_integrity(frames[t], fk_col, frames[parent], parent_pk)
            frames[t] = drop_null_context_fk(frames[t], fk_col, t)
    return frames


print("engine loaded: DatasetConfig/TableSpec + derivations + introspect + rekey + temporal + "
      "fk_remap(chain) + rarity(DP profiler, config_hash/RARITY_CONFIG_HASH) + synth_io "
      "(all cfg-first; bind cfg AFTER %run engine + %run <dataset>_config)")